Define question patterns

In [1]:
def identify_intent(question):
    q = question.strip().lower()

    if "දුම්රිය" in q or "කෝච්චි" in q:
        return "train_services"

    if "නවාතැන්" in q or "හෝටල්" in q:
        return "accommodation"

    if "කඳු නැග" in q or "තරණය" in q:
        return "hiking"

    if "පළාත" in q:
        return "province"

    if "දිස්ත්‍රික්ක" in q:
        return "district"

    if "උස" in q:
        if "සහ" in q or "අතරින්" in q:
            return "height_comparison"
        return "height"

    if "සුදුසු කාල" in q or "සුදුසු වේලාව" in q:
        return "visiting_time"

    return "unknown"

Test Sinhala questions

In [2]:
questions = [
    "ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?",
    "ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?",
    "ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?",
    "රාවණා ඇල්ලේ උස කීයද?",
    "රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?",
    "ඇල්ලට පැමිණෙන දුම්රිය සේවා මොනවාද?",
    "පුංචි ශ්‍රී පාදයට යාමට සුදුසු කාලය කුමක්ද?"
]

for question in questions:
    print("Question:", question)
    print("Detected intent:", identify_intent(question))
    print("-" * 50)

Question: ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?
Detected intent: province
--------------------------------------------------
Question: ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?
Detected intent: accommodation
--------------------------------------------------
Question: ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?
Detected intent: hiking
--------------------------------------------------
Question: රාවණා ඇල්ලේ උස කීයද?
Detected intent: height
--------------------------------------------------
Question: රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?
Detected intent: height_comparison
--------------------------------------------------
Question: ඇල්ලට පැමිණෙන දුම්රිය සේවා මොනවාද?
Detected intent: train_services
--------------------------------------------------
Question: පුංචි ශ්‍රී පාදයට යාමට සුදුසු කාලය කුමක්ද?
Detected intent: visiting_time
--------------------------------------------------


Sinhala Entity Identification
Goal: Python can identify what a user is asking about, such as height, hiking, or province. But it doesn't yet know which destination or attraction the question refers to.

In [2]:
entity_aliases = {
    "D001": ["ඇල්ල", "Ella"],

    "A001": [
        "පුංචි ශ්‍රී පාදය",
        "පුංචි සිරිපාදය",
        "Little Adam's Peak",
        "Mini Adam's Peak"
    ],

    "A002": ["ඇල්ල රොක්", "Ella Rock"],

    "A003": [
        "ආරුක්කු නවයේ පාලම",
        "Nine Arches Bridge"
    ],

    "A004": ["රාවණා ඇල්ල", "Ravana Falls"],

    "A005": ["රාවණා ගුහාව", "Ravana Cave"],

    "A006": ["දෝව රජමහා විහාරය"],

    "A007": [
        "දියලුම දිය ඇල්ල",
        "දියලුම ඇල්ල",
        "Diyaluma Falls"
    ],

    "A008": ["නිල්දිය පොකුණ"]
}

print("Entities configured:", len(entity_aliases))

Entities configured: 9


Implement entity matching
ex.ඇල්ල is contained inside රාවණා ඇල්ල, and matching the shorter name first would incorrectly detect two entities.

In [3]:
def identify_entities(question):
    text = question.casefold()

    candidates = []

    for entity_id, aliases in entity_aliases.items():
        for alias in aliases:
            alias_text = alias.casefold()

            start = text.find(alias_text)

            while start != -1:
                end = start + len(alias_text)

                candidates.append({
                    "id": entity_id,
                    "matched_text": question[start:end],
                    "start": start,
                    "end": end
                })

                start = text.find(alias_text, start + 1)

    # Match longer entity names first
    candidates.sort(
        key=lambda x: (
            -(x["end"] - x["start"]),
            x["start"]
        )
    )

    selected = []

    for candidate in candidates:
        overlaps = any(
            candidate["start"] < item["end"]
            and candidate["end"] > item["start"]
            for item in selected
        )

        if not overlaps:
            selected.append(candidate)

    selected.sort(key=lambda x: x["start"])

    return [
        {
            "id": item["id"],
            "matched_text": item["matched_text"]
        }
        for item in selected
    ]

In [17]:
def identify_intent(question):
    q = question.strip().lower()

    # Train services
    if any(word in q for word in ["දුම්රිය", "කෝච්චි"]):
        return "train_services"

    # Accommodation
    if any(word in q for word in ["නවාතැන්", "හෝටල්"]):
        return "accommodation"

    # Hiking
    if any(word in q for word in ["කඳු නැග", "තරණය"]):
        return "hiking"

    # Administrative locations
    if "පළාත" in q:
        return "province"

    if "දිස්ත්‍රික්ක" in q:
        return "district"

    # Visiting time
    if any(word in q for word in [
        "සුදුසු කාල",
        "සුදුසු වේලාව",
        "හොඳම වෙලාව",
        "හොඳම කාල",
        "යන්න හොඳ",
        "නැරඹීමට සුදුසු"
    ]):
        return "visiting_time"

    # Height comparison
    if "උස" in q:
        if any(word in q for word in [
            "සහ",
            "අතරින්",
            "වැඩි",
            "සැසඳ",
            "වඩා"
        ]):
            return "height_comparison"

        return "height"

    return "unknown"

Test entity identification

In [18]:
questions = [
    "ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?",
    "ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?",
    "ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?",
    "රාවණා ඇල්ලේ උස කීයද?",
    "රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?",
    "ඇල්ලට පැමිණෙන දුම්රිය සේවා මොනවාද?",
    "පුංචි ශ්‍රී පාදයට යාමට සුදුසු කාලය කුමක්ද?"
]

for question in questions:
    print("\nQuestion:", question)
    print("Intent:", identify_intent(question))
    print("Entities:", identify_entities(question))
    print("-" * 55)


Question: ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?
Intent: province
Entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
-------------------------------------------------------

Question: ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?
Intent: accommodation
Entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
-------------------------------------------------------

Question: ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?
Intent: hiking
Entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
-------------------------------------------------------

Question: රාවණා ඇල්ලේ උස කීයද?
Intent: height
Entities: [{'id': 'A004', 'matched_text': 'රාවණා ඇල්ල'}]
-------------------------------------------------------

Question: රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?
Intent: height_comparison
Entities: [{'id': 'A004', 'matched_text': 'රාවණා ඇල්ල'}, {'id': 'A007', 'matched_text': 'දියලුම ඇල්ල'}]
-------------------------------------------------------

Question: ඇල්ලට පැමිණෙන දුම්රිය සේවා මොනවාද?
Intent: train_services
Entities: [{'i

Combine intent and entities

In [19]:
def analyze_question(question):
    return {
        "question": question,
        "intent": identify_intent(question),
        "entities": identify_entities(question)
    }


test = analyze_question(
    "රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?"
)

print(test)

{'question': 'රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?', 'intent': 'height_comparison', 'entities': [{'id': 'A004', 'matched_text': 'රාවණා ඇල්ල'}, {'id': 'A007', 'matched_text': 'දියලුම ඇල්ල'}]}


Automatic Cypher Query Routing.
Goal: type a Sinhala question and have Python automatically select the correct Cypher query, retrieve the relevant facts from Neo4j, and display the result.

In [20]:
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase
import os

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

load_dotenv(project_root / ".env")

DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(
        os.getenv("NEO4J_USER"),
        os.getenv("NEO4J_PASSWORD")
    )
)

driver.verify_connectivity()

print("Connected to local Neo4j!")

Connected to local Neo4j!


Define the Cypher query templates

In [21]:
CYPHER_QUERIES = {
    "province": """
        MATCH (e:Destination {id: $entity_id})
              -[:LOCATED_IN]->(d:District)
              -[:PART_OF]->(p:Province)
        RETURN p.name_si AS province,
               d.name_si AS district
    """,

    "district": """
        MATCH (e:Destination {id: $entity_id})
              -[:LOCATED_IN]->(d:District)
        RETURN d.name_si AS district
    """,

    "accommodation": """
        MATCH (e:Destination {id: $entity_id})
              -[:HAS_ACCOMMODATION]->(h:Accommodation)
        RETURN h.name_si AS accommodation
        ORDER BY accommodation
    """,

    "hiking": """
        MATCH (e:Destination {id: $entity_id})
              -[:HAS_ATTRACTION]->(a:Attraction)
              -[:SUPPORTS_ACTIVITY]->(act:Activity {id: "AC001"})
        RETURN a.name_si AS hiking_location
        ORDER BY hiking_location
    """,

    "train_services": """
        MATCH (e:Destination {id: $entity_id})
              -[:HAS_RAILWAY_STATION]->(s:RailwayStation)
              <-[:ARRIVES_AT]-(t:TrainService)
        RETURN t.name_si AS train,
               t.train_number AS train_number,
               t.arrival_time AS document_arrival_time
        ORDER BY train_number
    """,

    "height": """
        MATCH (a:Attraction {id: $entity_id})
        RETURN a.name_si AS attraction,
               a.height_m AS height_m,
               a.height_approximate AS approximate
    """,

    "height_comparison": """
        MATCH (a:Attraction {id: $entity_id})
        MATCH (b:Attraction {id: $entity_id_2})
        WHERE a.height_m IS NOT NULL
          AND b.height_m IS NOT NULL
        RETURN a.name_si AS attraction_1,
               a.height_m AS height_1,
               b.name_si AS attraction_2,
               b.height_m AS height_2,
               CASE
                   WHEN a.height_m > b.height_m THEN a.name_si
                   WHEN b.height_m > a.height_m THEN b.name_si
                   ELSE "සමාන උස"
               END AS taller_attraction
    """,

    "visiting_time": """
        MATCH (a:Attraction {id: $entity_id})
        RETURN a.name_si AS attraction,
               a.best_time_si AS visiting_time
    """
}

print("Query templates:", len(CYPHER_QUERIES))

Query templates: 8


Automatically select and execute the query

Reads the Sinhala question.
Identifies its intent and entity IDs.
Selects a matching predefined Cypher query.
Executes the query in Neo4j.
Returns structured data that we can later pass to SinLlama

In [7]:
def retrieve_answer(question):
    analysis = analyze_question(question)

    intent = analysis["intent"]
    entities = analysis["entities"]

    print("Detected intent:", intent)
    print("Detected entities:", entities)

    if intent not in CYPHER_QUERIES:
        return {
            "status": "unsupported",
            "message": "මෙම ප්‍රශ්න වර්ගය තවම සහාය නොදක්වයි.",
            "data": []
        }

    if not entities:
        return {
            "status": "entity_not_found",
            "message": "ප්‍රශ්නයට අදාළ ස්ථානය හඳුනාගත නොහැකි විය.",
            "data": []
        }

    entity_ids = [item["id"] for item in entities]

    # Comparison questions require two different entities
    if intent == "height_comparison":
        if len(entity_ids) != 2:
            return {
                "status": "missing_entities",
                "message": "සැසඳීම සඳහා ස්ථාන දෙකක් අවශ්‍ය වේ.",
                "data": []
            }

        params = {
            "entity_id": entity_ids[0],
            "entity_id_2": entity_ids[1]
        }

    else:
        if len(entity_ids) != 1:
            return {
                "status": "ambiguous_entities",
                "message": "ප්‍රශ්නයේ ස්ථාන කිහිපයක් හඳුනාගෙන ඇත.",
                "data": []
            }

        params = {
            "entity_id": entity_ids[0]
        }

    query = CYPHER_QUERIES[intent]

    records, _, _ = driver.execute_query(
        query,
        parameters_=params,
        database_=DATABASE
    )

    results = [dict(record) for record in records]

    # Reject incomplete property evidence
    if intent == "height":
        results = [
            r for r in results
            if r.get("height_m") is not None
        ]

    elif intent == "visiting_time":
        results = [
            r for r in results
            if r.get("visiting_time") is not None
        ]

    if not results:
        return {
            "status": "no_evidence",
            "message": "අවශ්‍ය තොරතුරු Knowledge Graph තුළ නොමැත.",
            "data": []
        }

    return {
        "status": "success",
        "intent": intent,
        "entity_ids": entity_ids,
        "data": results
    }

Test the first automatic graph retrieval

In [22]:
question = "රාවණා ඇල්ලේ උස කීයද?"

result = retrieve_answer(question)

print("\nRetrieval result:")
print(result)

Detected intent: height
Detected entities: [{'id': 'A004', 'matched_text': 'රාවණා ඇල්ල'}]

Retrieval result:
{'status': 'success', 'intent': 'height', 'entity_ids': ['A004'], 'data': [{'attraction': 'රාවණා ඇල්ල', 'height_m': 25, 'approximate': True}]}


Test several Sinhala questions

In [23]:
test_questions = [
    "ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?",
    "ඇල්ල පිහිටා තිබෙන්නේ කුමන දිස්ත්‍රික්කයේද?",
    "ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?",
    "ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?",
    "රාවණා ඇල්ලේ උස කීයද?",
    "රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?",
    "ඇල්ලට පැමිණෙන දුම්රිය සේවා මොනවාද?",
    "පුංචි ශ්‍රී පාදයට යාමට සුදුසු කාලය කුමක්ද?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'=' * 60}")
    print(f"TEST {i}: {question}")

    result = retrieve_answer(question)

    print("Status:", result["status"])

    if result["status"] == "success":
        for record in result["data"]:
            print(record)
    else:
        print(result["message"])


TEST 1: ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?
Detected intent: province
Detected entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
Status: success
{'province': 'ඌව පළාත', 'district': 'බදුල්ල'}

TEST 2: ඇල්ල පිහිටා තිබෙන්නේ කුමන දිස්ත්‍රික්කයේද?
Detected intent: district
Detected entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
Status: success
{'district': 'බදුල්ල'}

TEST 3: ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?
Detected intent: accommodation
Detected entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
Status: success
{'accommodation': '98 Acres Resort & Spa'}
{'accommodation': 'Ella Jungle Resort'}
{'accommodation': 'La Ella Breeze'}
{'accommodation': 'The Secret Ella'}

TEST 4: ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?
Detected intent: hiking
Detected entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
Status: success
{'hiking_location': 'ඇල්ල රොක්'}
{'hiking_location': 'පුංචි ශ්\u200dරී පාදය'}

TEST 5: රාවණා ඇල්ලේ උස කීයද?
Detected intent: height
Detected entities: [{'id': 'A004', 'matched_text': 'ර

Create a test dataset
These 15 questions use information already represented in Ella Knowledge Graph, along with intentional unsupported or paraphrased cases.

In [24]:
evaluation_questions = [
    {
        "id": "T01",
        "question": "ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?",
        "expected_intent": "province",
        "expected_entities": ["D001"],
        "expected_status": "success"
    },
    {
        "id": "T02",
        "question": "ඇල්ල අයත් වන පළාත මොකක්ද?",
        "expected_intent": "province",
        "expected_entities": ["D001"],
        "expected_status": "success"
    },
    {
        "id": "T03",
        "question": "ඇල්ල තිබෙන්නේ මොන දිස්ත්‍රික්කයේද?",
        "expected_intent": "district",
        "expected_entities": ["D001"],
        "expected_status": "success"
    },
    {
        "id": "T04",
        "question": "ඇල්ලේ නවාතැන් ගන්න පුළුවන් තැන් මොනවාද?",
        "expected_intent": "accommodation",
        "expected_entities": ["D001"],
        "expected_status": "success"
    },
    {
        "id": "T05",
        "question": "ඇල්ලේ හෝටල් මොනවාද?",
        "expected_intent": "accommodation",
        "expected_entities": ["D001"],
        "expected_status": "success"
    },
    {
        "id": "T06",
        "question": "ඇල්ලේ කඳු නැගීමට පුළුවන් කොහේද?",
        "expected_intent": "hiking",
        "expected_entities": ["D001"],
        "expected_status": "success"
    },
    {
        "id": "T07",
        "question": "රාවණා ඇල්ලේ උස කීයද?",
        "expected_intent": "height",
        "expected_entities": ["A004"],
        "expected_status": "success"
    },
    {
        "id": "T08",
        "question": "රාවණා ඇල්ල මීටර් කීයක් උසද?",
        "expected_intent": "height",
        "expected_entities": ["A004"],
        "expected_status": "success"
    },
    {
        "id": "T09",
        "question": "රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?",
        "expected_intent": "height_comparison",
        "expected_entities": ["A004", "A007"],
        "expected_status": "success"
    },
    {
        "id": "T10",
        "question": "දියලුම ඇල්ලද රාවණා ඇල්ලද උසින් වැඩි?",
        "expected_intent": "height_comparison",
        "expected_entities": ["A007", "A004"],
        "expected_status": "success"
    },
    {
        "id": "T11",
        "question": "ඇල්ලට එන දුම්රිය මොනවාද?",
        "expected_intent": "train_services",
        "expected_entities": ["D001"],
        "expected_status": "success"
    },
    {
        "id": "T12",
        "question": "පුංචි ශ්‍රී පාදයට යන්න හොඳම වෙලාව කවදාද?",
        "expected_intent": "visiting_time",
        "expected_entities": ["A001"],
        "expected_status": "success"
    },
    {
        "id": "T13",
        "question": "ඇල්ලේ ජනගහනය කීයද?",
        "expected_intent": "unknown",
        "expected_entities": ["D001"],
        "expected_status": "unsupported"
    },
    {
        "id": "T14",
        "question": "සීගිරියේ උස කීයද?",
        "expected_intent": "height",
        "expected_entities": [],
        "expected_status": "entity_not_found"
    },
    {
        "id": "T15",
        "question": "ඇල්ලේ පාසල් කීයක් තිබෙනවාද?",
        "expected_intent": "unknown",
        "expected_entities": ["D001"],
        "expected_status": "unsupported"
    }
]

print("Total evaluation questions:", len(evaluation_questions))

Total evaluation questions: 15


Evaluate intent and entity identification

In [25]:
import pandas as pd

evaluation_results = []

for item in evaluation_questions:

    analysis = analyze_question(item["question"])

    predicted_intent = analysis["intent"]
    predicted_entities = [
        entity["id"] for entity in analysis["entities"]
    ]

    intent_correct = (
        predicted_intent == item["expected_intent"]
    )

    entity_correct = (
        predicted_entities == item["expected_entities"]
    )

    evaluation_results.append({
        "Question_ID": item["id"],
        "Question": item["question"],
        "Expected_Intent": item["expected_intent"],
        "Predicted_Intent": predicted_intent,
        "Intent_Correct": intent_correct,
        "Expected_Entities": item["expected_entities"],
        "Predicted_Entities": predicted_entities,
        "Entity_Correct": entity_correct
    })

question_eval_df = pd.DataFrame(evaluation_results)

display(question_eval_df)

,Question_ID,Question,Expected_Intent,Predicted_Intent,Intent_Correct,Expected_Entities,Predicted_Entities,Entity_Correct
0,T01,ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?,province,province,True,[D001],[D001],True
1,T02,ඇල්ල අයත් වන පළාත මොකක්ද?,province,province,True,[D001],[D001],True
2,T03,ඇල්ල තිබෙන්නේ මොන දිස්ත්‍රික්කයේද?,district,district,True,[D001],[D001],True
3,T04,ඇල්ලේ නවාතැන් ගන්න පුළුවන් තැන් මොනවාද?,accommodation,accommodation,True,[D001],[D001],True
4,T05,ඇල්ලේ හෝටල් මොනවාද?,accommodation,accommodation,True,[D001],[D001],True
5,T06,ඇල්ලේ කඳු නැගීමට පුළුවන් කොහේද?,hiking,hiking,True,[D001],[D001],True
6,T07,රාවණා ඇල්ලේ උස කීයද?,height,height,True,[A004],[A004],True
7,T08,රාවණා ඇල්ල මීටර් කීයක් උසද?,height,height,True,[A004],[A004],True
8,T09,රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කු...,height_comparison,height_comparison,True,"[A004, A007]","[A004, A007]",True
9,T10,දියලුම ඇල්ලද රාවණා ඇල්ලද උසින් වැඩි?,height_comparison,height_comparison,True,"[A007, A004]","[A007, A004]",True


Calculate accuracy

Intent accuracy-	Percentage of questions assigned the correct intent
Entity accuracy	-Percentage of questions with the exact expected entity IDs
Combined accuracy-	Percentage where both intent and entities were correct

In [26]:
intent_accuracy = question_eval_df["Intent_Correct"].mean()
entity_accuracy = question_eval_df["Entity_Correct"].mean()

combined_accuracy = (
    question_eval_df["Intent_Correct"]
    & question_eval_df["Entity_Correct"]
).mean()

print(f"Intent accuracy: {intent_accuracy:.2%}")
print(f"Entity accuracy: {entity_accuracy:.2%}")
print(f"Combined accuracy: {combined_accuracy:.2%}")

Intent accuracy: 100.00%
Entity accuracy: 100.00%
Combined accuracy: 100.00%


Display only errors

In [27]:
failed_questions = question_eval_df[
    (~question_eval_df["Intent_Correct"])
    | (~question_eval_df["Entity_Correct"])
]

print("Failed questions:", len(failed_questions))

display(
    failed_questions[
        [
            "Question_ID",
            "Question",
            "Expected_Intent",
            "Predicted_Intent",
            "Expected_Entities",
            "Predicted_Entities"
        ]
    ]
)

Failed questions: 0


,Question_ID,Question,Expected_Intent,Predicted_Intent,Expected_Entities,Predicted_Entities


Evaluate end-to-end graph retrieval status
Test automatic query routing

In [28]:
routing_results = []

for item in evaluation_questions:

    result = retrieve_answer(item["question"])

    routing_results.append({
        "Question_ID": item["id"],
        "Expected_Status": item["expected_status"],
        "Actual_Status": result["status"],
        "Status_Correct": (
            result["status"] == item["expected_status"]
        ),
        "Records_Returned": len(result["data"])
    })

routing_eval_df = pd.DataFrame(routing_results)

display(routing_eval_df)

print(
    "Routing status accuracy:",
    f"{routing_eval_df['Status_Correct'].mean():.2%}"
)

Detected intent: province
Detected entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
Detected intent: province
Detected entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
Detected intent: district
Detected entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
Detected intent: accommodation
Detected entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
Detected intent: accommodation
Detected entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
Detected intent: hiking
Detected entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
Detected intent: height
Detected entities: [{'id': 'A004', 'matched_text': 'රාවණා ඇල්ල'}]
Detected intent: height
Detected entities: [{'id': 'A004', 'matched_text': 'රාවණා ඇල්ල'}]
Detected intent: height_comparison
Detected entities: [{'id': 'A004', 'matched_text': 'රාවණා ඇල්ල'}, {'id': 'A007', 'matched_text': 'දියලුම ඇල්ල'}]
Detected intent: height_comparison
Detected entities: [{'id': 'A007', 'matched_text': 'දියලුම ඇල්ල'}, {'id': 'A004', 'matched_text': 'රාවණා ඇල්ල'}]


,Question_ID,Expected_Status,Actual_Status,Status_Correct,Records_Returned
0,T01,success,success,True,1
1,T02,success,success,True,1
2,T03,success,success,True,1
3,T04,success,success,True,4
4,T05,success,success,True,4
5,T06,success,success,True,2
6,T07,success,success,True,1
7,T08,success,success,True,1
8,T09,success,success,True,1
9,T10,success,success,True,1


Routing status accuracy: 100.00%


In [29]:
from pathlib import Path

output_dir = project_root / "results"
output_dir.mkdir(parents=True, exist_ok=True)

question_eval_df.to_csv(
    output_dir / "question_understanding_evaluation.csv",
    index=False,
    encoding="utf-8-sig"
)

routing_eval_df.to_csv(
    output_dir / "automatic_routing_evaluation.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Evaluation results saved successfully!")

Evaluation results saved successfully!


Improve the existing intent classifier